## Combining ACS 5 Year Metrics and Property Data

This notebook will require you to have ingested block groups (notebooks, 02_ingest, tiles, ingest_tiles), any parcel layer (notebooks, 02_ingest, parcels, ingest_parcels), and the census data (notebooks, 02_ingest, population, ingest_population). 

Here are the census metrics available to you: 

- b01001   # Sex by Age
- b19013   # Median Household Income
- b02001   # Race
- b25077   # Median Property Value (Dollars)
- b17001   # Poverty Status in the Past 12 Months
- b15003   # Educational attainment for over 25yo
- b23025   # Employment status (Labor force, unemployment) 
- b08303   # Travel time to work
- b25002   # Occupancy Status 
- b25064   # Median Gross Rent
- b25003   # Tenure (Owner vs Renter)
- b25091   # Mortgage Status by Selected Monthly Owner Costs
- b25070   # Gross Rent as % of Household Income (Rent Burden)
- b03002   # Hispanic or Latino Origin by Race
- c16002   # Household Language by English Proficiency
- b11004   # Family Type by Presence of Own Children
- b01003   # Total Population
- b22010   # Receipt of Food Stamps/SNAP

In [ ]:
import geopandas as gpd

### First, we import our ACS data for desired metrics. 

In [ ]:
# Get ACS data for selected metrics
from openplaces.api import get_dataset

med_household_inc = get_dataset(recipe='US_population-acs-2024', partition_id='b19013')
med_gross_rent = get_dataset(recipe='US_population-acs-2024', partition_id='b25064')
med_prop_val = get_dataset(recipe='US_population-acs-2024', partition_id='b25077')

### Next, we import our block group data, and merge on census_geo_id to the ACS metrics

In [ ]:
# Get block groups
from openplaces.api import get_entities

bg = get_entities(recipe='US_tile-census-2025_blockgroup', geom=True)
ma_bg = bg[bg['admin2_id'] == 'US-MA']

In [ ]:
gdf = (
    ma_bg.merge(med_household_inc, on='census_geo_id', how='left')
    .merge(med_gross_rent, on='census_geo_id', how='left')
    .merge(med_prop_val, on='census_geo_id', how='left')
)

In [ ]:
import numpy as np
import pandas as pd

cols = ['med_household_inc', 'med_gross_rent', 'med_prop_val']

# 1. ensure numeric
gdf[cols] = gdf[cols].apply(pd.to_numeric, errors='coerce')

# 2. remove ACS negative codes
gdf[cols] = gdf[cols].mask(gdf[cols] < 0, np.nan)

# 3. drop missing geometry (important for plotting stability)
gdf = gdf[gdf.geometry.notna()]
gdf = gdf[gdf.is_valid]

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(22, 7))

gdf.plot(column='med_household_inc', ax=axes[0], legend=True, cmap='Reds')
axes[0].set_title('Median Household Income')
axes[0].axis('off')

gdf.plot(column='med_gross_rent', ax=axes[1], legend=True, cmap='Greens')
axes[1].set_title('Median Gross Rent')
axes[1].axis('off')

gdf.plot(column='med_prop_val', ax=axes[2], legend=True, cmap='viridis_r')
axes[2].set_title('Median Property Value')
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Next, we can pull in our Massachusetts properties. 

In [ ]:
from openplaces.api import get_entities

prop = get_entities(
    recipe='US-MA_parcel-massgis-2025', admin_id='US-MA-SU', layer='property', geom=True
)
prop.plot()

### We can use a spatial join to connect our property attributes to census data. For a reminder on our tables... 

In [ ]:
gdf.head(1)

In [ ]:
prop.head(1)

In [ ]:
prop_census = gpd.sjoin(prop, gdf, how='left', predicate='intersects')

In [ ]:
ax = prop_census.plot(
    column='med_gross_rent', cmap='Blues', legend=True, figsize=(11, 11), alpha=0.5
)

prop_census['building_age'] = 2026 - prop_census['year_built']

prop_census.plot(column='building_age', cmap='Reds', markersize=2, alpha=0.6, ax=ax)

plt.axis('off')
plt.title('Housing Age vs Neighborhood Rent Levels')